In [7]:
!pip install rdflib

In [8]:
import json
import re
import time
import requests
import pandas as pd

from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD

In [ ]:
crawled_pages = []
with open("crawler_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        crawled_pages.append(json.loads(line))

entities_df = pd.read_csv("extracted_knowledge.csv")

print(f"Pages chargées       : {len(crawled_pages)}")
print(f"Entités chargées     : {len(entities_df)}")
print(f"Labels présents      : {entities_df['label'].value_counts().to_dict()}")
print(f"URLs uniques sources : {entities_df['source_url'].nunique()}")

Pages chargées       : 8
Entités chargées     : 2188
Labels présents      : {'ORG': 912, 'PERSON': 624, 'GPE': 416, 'DATE': 236}
URLs uniques sources : 8


In [ ]:
# Namespaces
EX  = Namespace("http://example.org/fifa/")
WD  = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

g = Graph()
g.bind("ex",   EX)
g.bind("wd",   WD)
g.bind("wdt",  WDT)
g.bind("owl",  OWL)
g.bind("rdfs", RDFS)

# Slugify — normalisation des URIs
def slugify(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s\-]+", "_", text)
    return text

def entity_uri(text: str) -> URIRef:
    return EX[slugify(text)]

def predicate_uri(text: str) -> URIRef:
    return EX[slugify(text)]

# Mapping label NER → classe RDF
label_to_class = {
    "PERSON": EX.Person,
    "ORG":    EX.Organization,
    "GPE":    EX.Place,
}


# --- Typage des entités depuis extracted_knowledge.csv ---
for _, row in entities_df.iterrows():
    ent_text  = str(row["entity"]).strip()
    ent_label = str(row["label"]).strip()

    if not ent_text or ent_label not in label_to_class:
        continue

    s = entity_uri(ent_text)
    g.add((s, RDF.type,    label_to_class[ent_label]))
    g.add((s, RDFS.label,  Literal(ent_text)))

# --- Triplets relationnels (nsubj/dobj du TD1) ---
import spacy
nlp = spacy.load("en_core_web_sm")

TARGET_LABELS = {"PERSON", "ORG", "GPE"}
triple_rows   = []

for page in crawled_pages:
    url  = page["url"]
    text = page["text"]
    doc  = nlp(text)

    for sent in doc.sents:
        sent_ents = [ent for ent in sent.ents if ent.label_ in TARGET_LABELS]
        if len(sent_ents) < 2:
            continue
        verbs = [tok for tok in sent if tok.pos_ == "VERB"]
        if not verbs:
            continue
        verb = verbs[0].lemma_.strip().lower()
        if not verb:
            continue
        subj = sent_ents[0].text.strip()
        obj  = sent_ents[1].text.strip()
        triple_rows.append({
            "subject":     subj,
            "predicate":   verb,
            "object":      obj,
            "source_url":  url
        })

triples_df = pd.DataFrame(triple_rows).drop_duplicates()
print(f"Triplets candidats : {len(triples_df)}")

for _, row in triples_df.iterrows():
    s = entity_uri(row["subject"])
    p = predicate_uri(row["predicate"])
    o = entity_uri(row["object"])
    g.add((s, p, o))

all_s = set(); all_o = set(); all_p = set()
for s, p, o in g:
    all_s.add(s); all_p.add(p)
    if isinstance(o, URIRef):
        all_o.add(o)

print(f"\n=== KB initiale ===")
print(f"Triplets  : {len(g)}")
print(f"Entités   : {len(all_s.union(all_o))}")
print(f"Relations : {len(all_p)}")

seuil_ok = True
if len(g) < 100:
    print("Attention < 100 triplets"); seuil_ok = False
if len(all_s.union(all_o)) < 50:
    print("Attention  < 50 entités"); seuil_ok = False
if seuil_ok:
    print("✅ Seuils Step 1 atteints")

Triplets candidats : 592

=== KB initiale ===
Triplets  : 3788
Entités   : 1713
Relations : 236
✅ Seuils Step 1 atteints


In [ ]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"

def search_wikidata(entity_name: str):
    params = {
        "action":   "wbsearchentities",
        "search":   entity_name,
        "language": "en",
        "format":   "json",
        "limit":    1
    }
    headers = {"User-Agent": "TD4-StudentProject/1.0"}
    try:
        r = requests.get(WIKIDATA_API, params=params, headers=headers, timeout=10)
        if r.status_code != 200 or not r.text.strip():
            return None, None
        data = r.json()
        results = data.get("search", [])
        if results:
            hit = results[0]
            uri = f"http://www.wikidata.org/entity/{hit['id']}"
            confidence = 0.9 if hit.get("match", {}).get("type") == "label" else 0.6
            return uri, confidence
    except Exception as e:
        print(f"Erreur pour '{entity_name}': {e}")
    return None, None

# Garder uniquement les entités les plus fréquentes (top 50)
top_entities = (
    entities_df[entities_df["label"].isin(["PERSON", "ORG", "GPE"])]
    .groupby("entity")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(50)
)

print(f"Entités à lier : {len(top_entities)} (les plus fréquentes)")

mapping_rows = []

for _, row in top_entities.iterrows():
    ent_text    = str(row["entity"]).strip()
    private_uri = entity_uri(ent_text)

    ext_uri, confidence = search_wikidata(ent_text)

    if ext_uri and confidence >= 0.7:
        g.add((private_uri, OWL.sameAs, URIRef(ext_uri)))
        mapping_rows.append({
            "private_entity": str(private_uri),
            "external_uri":   ext_uri,
            "confidence":     confidence,
            "status":         "linked"
        })
    else:
        mapping_rows.append({
            "private_entity": str(private_uri),
            "external_uri":   None,
            "confidence":     confidence if confidence else 0.0,
            "status":         "new_entity"
        })

    time.sleep(0.3)

mapping_df = pd.DataFrame(mapping_rows)
linked   = mapping_df[mapping_df["status"] == "linked"]
new_ents = mapping_df[mapping_df["status"] == "new_entity"]

print(f"Entités liées     : {len(linked)}")
print(f"Nouvelles entités : {len(new_ents)}")
print("\n--- Mapping table ---")
print(mapping_df.to_string(index=False))

Entités à lier : 50 (les plus fréquentes)
Entités liées     : 35
Nouvelles entités : 15

--- Mapping table ---
                                                           private_entity                             external_uri  confidence     status
                                          http://example.org/fifa/england       http://www.wikidata.org/entity/Q21         0.9     linked
                                             http://example.org/fifa/fifa   http://www.wikidata.org/entity/Q253414         0.9     linked
                                          http://example.org/fifa/ireland       http://www.wikidata.org/entity/Q27         0.9     linked
                                            http://example.org/fifa/wales       http://www.wikidata.org/entity/Q25         0.9     linked
                                http://example.org/fifa/the_united_states                                     None         0.6 new_entity
                                      http://example.org/fifa

In [14]:
g.add((EX.Person,       RDF.type,        OWL.Class))
g.add((EX.Person,       RDFS.subClassOf, URIRef("http://schema.org/Person")))
g.add((EX.Organization, RDF.type,        OWL.Class))
g.add((EX.Organization, RDFS.subClassOf, URIRef("http://schema.org/Organization")))
g.add((EX.Place,        RDF.type,        OWL.Class))
g.add((EX.Place,        RDFS.subClassOf, URIRef("http://schema.org/Place")))

for pred in all_p:
    g.add((pred, RDF.type, OWL.ObjectProperty))

for _, row in new_ents.iterrows():
    private_uri = URIRef(row["private_entity"])
    g.add((private_uri, RDFS.comment, Literal("Entity not found in Wikidata — defined in private KB")))

print("✅ Définitions ontologiques ajoutées")

✅ Définitions ontologiques ajoutées


In [ ]:
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

def find_wikidata_predicate(keyword: str):
    query = f"""
    SELECT ?property ?propertyLabel WHERE {{
      ?property a wikibase:Property .
      ?property rdfs:label ?propertyLabel .
      FILTER(CONTAINS(LCASE(?propertyLabel), "{keyword}"))
      FILTER(LANG(?propertyLabel) = "en")
    }}
    LIMIT 5
    """
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "TD4-StudentProject/1.0"
    }
    try:
        r = requests.get(WIKIDATA_SPARQL, params={"query": query},
                         headers=headers, timeout=15)
        if r.status_code != 200 or not r.text.strip():
            return []
        data = r.json()
        return [(b["property"]["value"], b["propertyLabel"]["value"])
                for b in data["results"]["bindings"]]
    except Exception as e:
        print(f"Erreur SPARQL '{keyword}': {e}")
        return []

# Top 8 prédicats les plus fréquents dans les triplets extraits
top_predicates = (
    triples_df["predicate"]
    .value_counts()
    .head(8)
    .index.tolist()
)

print("=== Candidats Wikidata pour chaque prédicat privé ===\n")
for verb in top_predicates:
    results = find_wikidata_predicate(verb)
    print(f"'{verb}' → candidats :")
    for uri, label in results[:5]:
        print(f"   {uri.split('/')[-1]}  →  {label}")
    print()
    time.sleep(1.5) 

=== Candidats Wikidata pour chaque prédicat privé ===

'retrieve' → candidats :
   P813  →  retrieved

'say' → candidats :
   P2268  →  Musée d'Orsay artist or personality ID
   P4659  →  Musée d'Orsay artwork ID
   P9470  →  HistoryLink essay number
   P12979  →  Say Who ID
   P12325  →  Sayed Ganj Balochi Glossary ID

'hold' → candidats :
   P2098  →  substitute/deputy/replacement of office/officeholder
   P2220  →  household wealth
   P5020  →  Code of Household Registration and Conscription Information System (Taiwan)
   P3931  →  copyright holder
   P1308  →  position holder

'archive' → candidats :
   P724  →  Internet Archive ID
   P599  →  International Tennis Federation player ID before 2020 (archived)
   P485  →  archives at
   P4067  →  COSR athlete ID (archived)
   P4068  →  Chinese Olympic Committee athlete ID (archived)

'win' → candidats :
   P190  →  twinned administrative body
   P537  →  twinning
   P600  →  Wine AppDB ID
   P2050  →  wingspan
   P2091  →  World Rowin

In [17]:
predicate_alignment = {
    "win":       "http://www.wikidata.org/prop/direct/P166",  # award received
    "organise":  "http://www.wikidata.org/prop/direct/P664",  # organizer of
    "include":   "http://www.wikidata.org/prop/direct/P527",  # has part
    "hold":      "http://www.wikidata.org/prop/direct/P664",  # organizer of
    "represent": "http://www.wikidata.org/prop/direct/P1268", # represents
    "found":     "http://www.wikidata.org/prop/direct/P571",  # inception
    "govern":    "http://www.wikidata.org/prop/direct/P748",  # appointed by
    "comprise":  "http://www.wikidata.org/prop/direct/P527",  # has part
}

WDT = Namespace("http://www.wikidata.org/prop/direct/")
g.bind("wdt", WDT)

for private_verb, wikidata_uri in predicate_alignment.items():
    private_pred = predicate_uri(private_verb)
    g.add((private_pred, OWL.equivalentProperty, URIRef(wikidata_uri)))

print(f"✅ {len(predicate_alignment)} alignements de prédicats ajoutés")

✅ 8 alignements de prédicats ajoutés


In [ ]:
def expand_1hop(wikidata_id: str, limit: int = 1000):
    query = f"""
    SELECT ?p ?o WHERE {{
      wd:{wikidata_id} ?p ?o .
    }}
    LIMIT {limit}
    """
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "TD4-StudentProject/1.0"
    }
    try:
        r = requests.get(WIKIDATA_SPARQL, params={"query": query},
                         headers=headers, timeout=30)
        if r.status_code != 200 or not r.text.strip():
            return []
        data = r.json()
        return [(b["p"]["value"], b["o"]["value"])
                for b in data["results"]["bindings"]]
    except Exception as e:
        print(f"Erreur expansion {wikidata_id}: {e}")
        return []

aligned_entities = mapping_df[mapping_df["confidence"] >= 0.7].copy()
print(f"Entités alignées disponibles pour expansion : {len(aligned_entities)}\n")

expansion_count = 0

for _, row in aligned_entities.iterrows():
    ext_uri = row["external_uri"]
    if not ext_uri:
        continue

    wikidata_id = ext_uri.split("/")[-1]
    triples_exp = expand_1hop(wikidata_id, limit=1000)

    for p_uri, o_val in triples_exp:
        subj = URIRef(ext_uri)
        pred = URIRef(p_uri)
        obj  = URIRef(o_val) if o_val.startswith("http") else Literal(o_val)
        g.add((subj, pred, obj))
        expansion_count += 1

    print(f"  {wikidata_id} : +{len(triples_exp)} triplets")
    time.sleep(1.5)  # rate limit

print(f"\n✅ Expansion terminée")
print(f"Triplets ajoutés par expansion : {expansion_count}")
print(f"Taille totale du graphe        : {len(g)} triplets")

# Vérification seuils Step 4
if len(g) < 50_000:
    print("Attention < 50 000 triplets — augmente limit dans expand_1hop ou ajoute des entités alignées")
elif len(g) > 200_000:
    print("Attention  > 200 000 triplets — réduis limit dans expand_1hop")
else:
    print("✅ Seuil triplets Step 4 OK")

Entités alignées disponibles pour expansion : 35

  Q21 : +1000 triplets
  Q253414 : +644 triplets
  Q27 : +1000 triplets
  Q25 : +1000 triplets
  Q39 : +1000 triplets
  Q159 : +1000 triplets
  Q160549 : +431 triplets
  Q58733 : +352 triplets
  Q84 : +1000 triplets
  Q26 : +905 triplets
  Q22 : +1000 triplets
  Q142 : +1000 triplets
  Q155 : +1000 triplets
  Q1340204 : +162 triplets
  Q124138 : +583 triplets
  Q34 : +1000 triplets
  Q8798 : +800 triplets
  Q552359 : +110 triplets
  Q45 : +1000 triplets
  Q38 : +1000 triplets
  Q11148 : +418 triplets
  Q40469 : +604 triplets
  Q90 : +1000 triplets
  Q9684 : +656 triplets
  Q130879 : +451 triplets
  Q35 : +1000 triplets
  Q217776 : +282 triplets
  Q483437 : +512 triplets
  Q21476697 : +585 triplets
  Q29 : +1000 triplets
  Q253414 : +644 triplets
  Q20 : +1000 triplets
  Q212 : +1000 triplets
  Q801 : +1000 triplets
  Q43 : +1000 triplets

✅ Expansion terminée
Triplets ajoutés par expansion : 27139
Taille totale du graphe        : 25091 

In [ ]:
before = len(g)

to_remove = [
    (s, p, o) for s, p, o in g
    if (isinstance(s, URIRef) and not str(s).strip()) or
       (isinstance(o, URIRef) and not str(o).strip())
]
for triple in to_remove:
    g.remove(triple)

NOISY_PREDICATES = [
    URIRef("http://schema.org/description"),
    URIRef("http://www.w3.org/2000/01/rdf-schema#comment"),
    URIRef("http://www.w3.org/2004/02/skos/core#altLabel"),
]
for pred in NOISY_PREDICATES:
    for triple in list(g.triples((None, pred, None))):
        g.remove(triple)

after = len(g)

# Stats finales
all_s_f = set(); all_o_f = set(); all_p_f = set()
for s, p, o in g:
    all_s_f.add(s); all_p_f.add(p)
    if isinstance(o, URIRef):
        all_o_f.add(o)

final_triples   = len(g)
final_entities  = len(all_s_f.union(all_o_f))
final_relations = len(all_p_f)

print(f"Triplets avant nettoyage : {before}")
print(f"Triplets supprimés       : {before - after}")
print(f"\n=== Stats finales KB ===")
print(f"Triplets  : {final_triples}")
print(f"Entités   : {final_entities}")
print(f"Relations : {final_relations}")

# Vérification seuils entités
if final_entities < 5_000:
    print("Attention  Entités < 5 000")
elif final_entities > 30_000:
    print("Attention  Entités > 30 000")
else:
    print("✅ Seuil entités Step 4 OK")

Triplets avant nettoyage : 25091
Triplets supprimés       : 5576

=== Stats finales KB ===
Triplets  : 19515
Entités   : 8936
Relations : 1729
✅ Seuil entités Step 4 OK


In [20]:
# 1. KB étendue
g.serialize(destination="expanded_kb.nt", format="nt")
print("✅ expanded_kb.nt")

# 2. Ontologie
g_onto = Graph()
g_onto.bind("ex", EX); g_onto.bind("owl", OWL); g_onto.bind("rdfs", RDFS)
onto_predicates = {RDF.type, RDFS.subClassOf, OWL.equivalentProperty,
                   RDFS.domain, RDFS.range, RDFS.comment}
for s, p, o in g:
    if p in onto_predicates:
        g_onto.add((s, p, o))
g_onto.serialize(destination="ontology.ttl", format="turtle")
print("✅ ontology.ttl")

# 3. Alignement
g_align = Graph()
g_align.bind("ex", EX); g_align.bind("owl", OWL)
for s, p, o in g:
    if p in {OWL.sameAs, OWL.equivalentProperty}:
        g_align.add((s, p, o))
g_align.serialize(destination="alignment.ttl", format="turtle")
print("✅ alignment.ttl")

# 4. Mapping table CSV
mapping_df.to_csv("alignment_mapping.csv", index=False, encoding="utf-8")
print("✅ alignment_mapping.csv")

# 5. Statistics report
stats = {
    "triplets":            final_triples,
    "entities":            final_entities,
    "relations":           final_relations,
    "linked_entities":     int(len(linked)),
    "new_entities":        int(len(new_ents)),
    "aligned_predicates":  len(predicate_alignment)
}
with open("statistics_report.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)
print("✅ statistics_report.json")
print(f"\nRécapitulatif final : {stats}")

/usr/local/lib/python3.12/dist-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


✅ expanded_kb.nt
✅ ontology.ttl
✅ alignment.ttl
✅ alignment_mapping.csv
✅ statistics_report.json

Récapitulatif final : {'triplets': 19515, 'entities': 8936, 'relations': 1729, 'linked_entities': 35, 'new_entities': 15, 'aligned_predicates': 8}
